In [0]:
# ============================================================
# NOTEBOOK: silver_produtos_categorias
# Squad 1 · Dupla 3 — Bruna Diniz & Rafael Miranda
# ============================================================
    
%pip install deltalake --quiet
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F
from pyspark.sql import Window
from deltalake import DeltaTable, write_deltalake

load_dotenv("/Workspace/Repos/diniz.dbruna@gmail.com/estagio-empregadados-turma-2/.env")

TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")

STORAGE_ACCOUNT = "internshipdatalake"
SQUAD_CONTAINER = "squad1"
lakehouse_base = f"abfss://{SQUAD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

bronze_produtos_path = f"{lakehouse_base}/bronze/ecommerce_produtos"
bronze_categorias_path = f"{lakehouse_base}/bronze/ecommerce_categorias"
silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": CLIENT_ID,
    "azure_storage_client_secret": CLIENT_SECRET,
    "azure_storage_tenant_id": TENANT_ID,
}

def ler_delta_direto(caminho):
    dt = DeltaTable(caminho, storage_options=storage_options)
    return spark.createDataFrame(dt.to_pandas())

print("Setup concluído.")

In [0]:
def regras_categorias(df):
    df_ref = df.select("bronze_source_file", "id_categoria", "id_categoria_pai").dropDuplicates(["bronze_source_file", "id_categoria"])

    w_dup = Window.partitionBy("bronze_source_file", "id_categoria")
    df = df.withColumn("_id_count", F.count("id_categoria").over(w_dup))

    df = (df
        .withColumn("regra_01_id_valido",
            F.col("id_categoria").isNotNull() & (F.col("_id_count") == 1))
        .withColumn("regra_02_nome_preenchido",
            F.col("nome_categoria").isNotNull() & (F.trim(F.col("nome_categoria")) != ""))
        .withColumn("regra_04_sem_ciclo_direto",
            F.col("id_categoria_pai").isNull() | (F.col("id_categoria_pai") != F.col("id_categoria")))
        .withColumn("regra_09_nome_sem_caracteres_invalidos",
            ~F.col("nome_categoria").rlike(r'[<>]|script'))
    ).drop("_id_count")

    w_nivel = Window.partitionBy("bronze_source_file", F.coalesce(F.col("id_categoria_pai"), F.lit(-1)), "nome_categoria")
    df = df.withColumn("_nome_nivel_count", F.count("nome_categoria").over(w_nivel))
    df = df.withColumn("regra_05_nome_unico_no_nivel", F.col("_nome_nivel_count") == 1).drop("_nome_nivel_count")

    ids_por_arquivo = df_ref.groupBy("bronze_source_file").agg(F.collect_set("id_categoria").alias("_ids_validos"))
    df = df.join(ids_por_arquivo, "bronze_source_file", "left")
    df = df.withColumn(
        "regra_03_pai_existe",
        F.col("id_categoria_pai").isNull() | F.array_contains(F.col("_ids_validos"), F.col("id_categoria_pai"))
    ).drop("_ids_validos")

    df_pais = df_ref.select(
        F.col("bronze_source_file").alias("_pf"),
        F.col("id_categoria").alias("_id"),
        F.col("id_categoria_pai").alias("_pai_do_pai")
    )
    df = (df
        .join(df_pais, (df["bronze_source_file"] == F.col("_pf")) & (df["id_categoria_pai"] == F.col("_id")), "left")
        .withColumn("regra_06_dois_niveis",
            F.col("id_categoria_pai").isNull() | F.col("_pai_do_pai").isNull())
        .drop("_pf", "_id", "_pai_do_pai")
    )

    subcats_por_arquivo = (df_ref.filter(F.col("id_categoria_pai").isNotNull())
        .groupBy("bronze_source_file").agg(F.collect_set("id_categoria_pai").alias("_pais_com_filho")))
    df = df.join(subcats_por_arquivo, "bronze_source_file", "left")
    df = df.withColumn(
        "regra_08_raiz_tem_subcategoria",
        F.when(F.col("id_categoria_pai").isNotNull(), F.lit(True))
         .otherwise(F.array_contains(F.col("_pais_com_filho"), F.col("id_categoria")))
    ).drop("_pais_com_filho")

    df = df.withColumn("regra_07_subcategoria_tem_produto", F.lit(None).cast("boolean"))
    df = df.withColumn("regra_10_total_raizes_confere", F.lit(None).cast("boolean"))

    return df

In [0]:
def regras_produtos(df, df_categorias_silver):
    df_cat_ref = df_categorias_silver.select("id_categoria", "id_categoria_pai").dropDuplicates(["id_categoria"])

    w_sku = Window.partitionBy("bronze_source_file", "sku")
    df = df.withColumn("_sku_count", F.count("sku").over(w_sku))

    df = (df
        .withColumn("regra_01_sku_valido",
            F.col("sku").isNotNull() & (F.col("_sku_count") == 1))
        .withColumn("regra_02_preco_positivo",
            F.col("preco_lista").isNotNull() & (F.col("preco_lista") > 0))
        .withColumn("regra_03_nome_preenchido",
            F.col("nome_produto").isNotNull() & (F.trim(F.col("nome_produto")) != ""))
        .withColumn("regra_05_unidade_valida",
            F.col("unidade_medida").isin("un", "kg", "g", "L", "ml"))
        .withColumn("regra_06_is_ativo_valido",
            F.col("is_ativo").isNotNull())
        .withColumn("regra_07_preco_dentro_limite",
            F.col("preco_lista").isNotNull() & (F.col("preco_lista") <= 5000))
        .withColumn("regra_08_marca_preenchida",
            F.col("nome_marca").isNotNull() & (F.trim(F.col("nome_marca")) != ""))
    ).drop("_sku_count")

    df = (df
        .join(df_cat_ref, "id_categoria", "left")
        .withColumn("regra_04_categoria_existe", F.col("id_categoria_pai").isNotNull() | F.col("id_categoria").isNotNull())
        .withColumn("regra_10_pertence_subcategoria", F.col("id_categoria_pai").isNotNull())
    )

    # PENDENTE (provisório até implementarmos a leitura da Raw, ver tarefa dedicada)
    df = df.withColumn("regra_09_ativo_com_venda_90d", F.lit(None).cast("boolean"))

    return df

In [0]:
def processar_silver(caminho_bronze, caminho_silver, aplicar_regras):
    df_bronze = ler_delta_direto(caminho_bronze)

    try:
        dt_silver = DeltaTable(caminho_silver, storage_options=storage_options)
        ja_processados = set(dt_silver.to_pandas(columns=["bronze_source_file"])["bronze_source_file"].unique())
        df_micro_lote = df_bronze.filter(~F.col("bronze_source_file").isin(ja_processados))
    except Exception as e:
        print(f"[AVISO] Não foi possível ler Silver existente em {caminho_silver}: {e}")
        print("[AVISO] Tratando TODO o conteúdo da Bronze como pendente. Confirme se isso é esperado!")
        df_micro_lote = df_bronze

    df_silver = aplicar_regras(df_micro_lote).withColumn(
        "silver_processed_at", F.current_timestamp()
    )

    df_pandas = df_silver.toPandas()
    df_pandas["bronze_ingested_at"] = df_pandas["bronze_ingested_at"].dt.tz_localize("UTC")
    df_pandas["silver_processed_at"] = df_pandas["silver_processed_at"].dt.tz_localize("UTC")

    for coluna in df_pandas.columns:
        if coluna.startswith("regra_"):
            df_pandas[coluna] = df_pandas[coluna].astype("boolean")

    write_deltalake(caminho_silver, df_pandas, mode="append", storage_options=storage_options)

    print(f"OK [{caminho_silver}] {len(df_pandas)} linhas processadas")
    return df_silver

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
datalake_service = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential
)
squad1_fs_client = datalake_service.get_file_system_client(SQUAD_CONTAINER)

for caminho in ["silver/ecommerce_produtos", "silver/ecommerce_categorias"]:
    try:
        squad1_fs_client.get_directory_client(caminho).delete_directory()
        print(f"Apagado: {caminho}")
    except Exception as e:
        print(f"{caminho}: nada pra apagar ou erro -> {e}")

In [0]:
processar_silver(bronze_categorias_path, silver_categorias_path, regras_categorias)
df_categorias_silver = ler_delta_direto(silver_categorias_path)
processar_silver(bronze_produtos_path, silver_produtos_path, lambda df: regras_produtos(df, df_categorias_silver))

In [0]:
print("Bronze produtos:", ler_delta_direto(bronze_produtos_path).count())
print("Bronze categorias:", ler_delta_direto(bronze_categorias_path).count())
print("Silver produtos:", DeltaTable(silver_produtos_path, storage_options=storage_options).to_pandas().shape[0])
print("Silver categorias:", DeltaTable(silver_categorias_path, storage_options=storage_options).to_pandas().shape[0])